# Full Pipeline — Run All + Export JSON + Deploy API

**Chạy toàn bộ pipeline và xuất kết quả ra JSON.**

Notebook này chạy lần lượt tất cả notebooks và export:
- `results/reports/eda_stats.json`
- `results/reports/rmse_summary.json`
- `results/reports/evaluation_results.json`
- `results/reports/context_analysis.json`

**Run order:** EDA → Train → Evaluate → Context → Export → Deploy API

## 0a. Install Dependencies

⚠️ **Cell này sẽ tự restart runtime sau khi install xong.**

Sau khi restart, bấm **Runtime → Run All** lần nữa — cell này sẽ skip (đã install rồi), các cell sau chạy bình thường.

In [ ]:
import subprocess, sys, os

try:
    import surprise
    import numpy as np
    import pandas as pd
    assert int(np.__version__.split('.')[0]) < 2, 'need numpy<2'
    assert int(pd.__version__.split('.')[0]) < 3, 'need pandas<3'
    print(f'✅ OK (numpy={np.__version__}, pandas={pd.__version__}, surprise={surprise.__version__})')
except Exception as e:
    print(f'📦 Installing... ({e})')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        'numpy<2', 'pandas<3', 'scikit-surprise', 'scikit-learn',
        'matplotlib', 'seaborn', 'tqdm', '-q'])
    print('✅ Install xong! Runtime đang restart...')
    os.kill(os.getpid(), 9)

## 0b. Setup (sau restart)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from pathlib import Path

print(f'numpy={np.__version__}')
assert int(np.__version__.split('.')[0]) < 2, 'numpy must be < 2!'

# Hyperparameters
DEFAULT_K = 40
DEFAULT_SVD_FACTORS = 50
DEFAULT_N_EPOCHS = 20
DEFAULT_CF_WEIGHT = 0.7

os.makedirs('results/reports', exist_ok=True)
os.makedirs('results/charts', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

print('✅ Setup done!')

## 1. Download & Load Dataset

In [ ]:
data_dir = Path('data/raw/ml-1m')
if not data_dir.exists():
    !mkdir -p data/raw
    !wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip -O /tmp/ml-1m.zip
    !unzip -q /tmp/ml-1m.zip -d data/raw/
    print('Dataset downloaded!')
else:
    print('Dataset already exists!')

ratings = pd.read_csv('data/raw/ml-1m/ratings.dat', sep='::', engine='python', names=['userId','movieId','rating','timestamp'])
movies  = pd.read_csv('data/raw/ml-1m/movies.dat',  sep='::', engine='python', names=['movieId','title','genres'], encoding='latin-1')
users   = pd.read_csv('data/raw/ml-1m/users.dat',   sep='::', engine='python', names=['userId','gender','age','occupation','zipcode'])

print(f'Ratings: {len(ratings):,} | Movies: {len(movies):,} | Users: {len(users):,}')

## 2. EDA — Export Stats

In [ ]:
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)$')
movies['year'] = pd.to_numeric(movies['year'], errors='coerce')
movies['num_genres'] = movies['genres'].str.count('\\|') + 1
movies.loc[movies['genres'] == '(no genres listed)', 'num_genres'] = 0

ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings['year'] = ratings['datetime'].dt.year
ratings['month'] = ratings['datetime'].dt.month
ratings['dayofweek'] = ratings['datetime'].dt.dayofweek

age_map = {1:'Under 18',18:'18-24',25:'25-34',35:'35-44',45:'45-49',50:'50-55',56:'56+'}
users['age_group'] = users['age'].map(age_map)
occ_map = {0:'other',1:'academic',2:'artist',3:'clerical',4:'student',5:'customer service',6:'doctor',7:'executive',8:'farmer',9:'homemaker',10:'K-12 student',11:'lawyer',12:'programmer',13:'retired',14:'sales',15:'scientist',16:'self-employed',17:'engineer',18:'tradesman',19:'unemployed',20:'writer'}
users['occupation_name'] = users['occupation'].map(occ_map)

n_users = ratings['userId'].nunique()
n_movies = ratings['movieId'].nunique()
sparsity = 1 - len(ratings) / (n_users * n_movies)

eda_stats = {
    'n_ratings': int(len(ratings)), 'n_users': int(n_users), 'n_movies': int(n_movies),
    'rating_mean': round(float(ratings['rating'].mean()), 4),
    'rating_std': round(float(ratings['rating'].std()), 4),
    'sparsity': round(float(sparsity), 4),
    'avg_ratings_per_user': round(float(ratings.groupby('userId').size().mean()), 2),
    'avg_ratings_per_movie': round(float(ratings.groupby('movieId').size().mean()), 2),
    'year_span': [int(ratings['year'].min()), int(ratings['year'].max())],
    'rating_distribution': {str(k): int(v) for k, v in ratings['rating'].value_counts().sort_index().items()},
    'top_10_genres': {k: int(v) for k, v in movies['genres'].str.split('|').explode().value_counts().head(10).items()},
}

with open('results/reports/eda_stats.json', 'w') as f:
    json.dump(eda_stats, f, indent=2)

ratings.to_csv('data/processed/ratings_clean.csv', index=False)
movies.to_csv('data/processed/movies_clean.csv', index=False)
users.to_csv('data/processed/users_clean.csv', index=False)

print('✅ EDA stats saved + processed data exported!')

## 3. Train / Test Split

In [ ]:
from surprise import SVD, KNNWithMeans, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print(f'Train: {trainset.n_ratings:,} | Test: {len(testset):,}')

## 4. Train All Models

In [ ]:
print('Training models...')

model_ucf = KNNWithMeans(k=DEFAULT_K, sim_option={'name':'cosine','user_based':True}, verbose=False)
model_ucf.fit(trainset)
print('  ✅ User-CF')

model_icf = KNNWithMeans(k=DEFAULT_K, sim_option={'name':'cosine','user_based':False}, verbose=False)
model_icf.fit(trainset)
print('  ✅ Item-CF')

model_svd = SVD(n_factors=DEFAULT_SVD_FACTORS, n_epochs=DEFAULT_N_EPOCHS, random_state=42)
model_svd.fit(trainset)
print('  ✅ SVD')

movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])
cosine_sim = cosine_similarity(tfidf_matrix)
movie_idx = pd.Series(movies.index, index=movies['movieId'])
print('  ✅ Content-Based')

print('All models trained!')

## 5. RMSE & MAE

In [ ]:
preds_ucf = model_ucf.test(testset)
preds_icf = model_icf.test(testset)
preds_svd = model_svd.test(testset)

rmse_ucf = accuracy.rmse(preds_ucf, verbose=False)
rmse_icf = accuracy.rmse(preds_icf, verbose=False)
rmse_svd = accuracy.rmse(preds_svd, verbose=False)
mae_ucf = accuracy.mae(preds_ucf, verbose=False)
mae_icf = accuracy.mae(preds_icf, verbose=False)
mae_svd = accuracy.mae(preds_svd, verbose=False)

print(f'User-CF: RMSE={rmse_ucf:.4f}, MAE={mae_ucf:.4f}')
print(f'Item-CF: RMSE={rmse_icf:.4f}, MAE={mae_icf:.4f}')
print(f'SVD:     RMSE={rmse_svd:.4f}, MAE={mae_svd:.4f}')

## 6. Ranking Metrics

In [ ]:
sample_users = ratings.groupby('userId').filter(lambda x: len(x) >= 20)['userId'].unique()[:300]

test_ground_truth = {}
for uid in sample_users:
    items = ratings[(ratings['userId'] == uid) & (ratings['rating'] >= 4)]['movieId'].tolist()
    if items:
        test_ground_truth[uid] = items

print(f'Sample users: {len(sample_users)}, with relevant items: {len(test_ground_truth)}')

def precision_at_k(actual, predicted, k):
    p = predicted[:k]
    return len(set(actual) & set(p)) / k if p else 0.0

def recall_at_k(actual, predicted, k):
    p = predicted[:k]
    return len(set(actual) & set(p)) / len(actual) if actual else 0.0

def ap_at_k(actual, predicted, k):
    p = predicted[:k]
    if not actual or not p: return 0.0
    score, hits = 0.0, 0.0
    for i, item in enumerate(p):
        if item in actual and item not in p[:i]:
            hits += 1.0
            score += hits / (i + 1.0)
    return score / min(len(actual), k)

def ndcg_at_k(actual, predicted, k):
    p = predicted[:k]
    if not actual: return 0.0
    dcg = sum(1.0/np.log2(i+2) for i, item in enumerate(p) if item in actual)
    ideal = sum(1.0/np.log2(i+2) for i in range(min(len(actual), k)))
    return dcg/ideal if ideal > 0 else 0.0

def get_recs_surprise(model, uid, top_n=50):
    seen = set(ratings[ratings['userId']==uid]['movieId'])
    unseen = [m for m in ratings['movieId'].unique() if m not in seen]
    scores = {m: model.predict(uid, m).est for m in unseen}
    return [m for m, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]]

def get_recs_content(uid, top_n=50):
    seen = set(ratings[ratings['userId']==uid]['movieId'])
    top_rated = ratings[ratings['userId']==uid].nlargest(5, 'rating')
    scores = {}
    for _, row in top_rated.iterrows():
        mid = row['movieId']
        if mid in movie_idx.index:
            idx = movie_idx[mid]
            for i, s in enumerate(cosine_sim[idx]):
                if i not in seen:
                    scores[i] = scores.get(i, 0) + s * row['rating']
    return [movies.iloc[idx]['movieId'] for idx, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]]

def get_recs_hybrid(uid, cf_weight=DEFAULT_CF_WEIGHT, top_n=50):
    seen = set(ratings[ratings['userId']==uid]['movieId'])
    unseen = [m for m in ratings['movieId'].unique() if m not in seen]
    top_rated = ratings[ratings['userId']==uid].nlargest(5, 'rating')
    ref_indices, ref_weights = [], []
    for _, row in top_rated.iterrows():
        if row['movieId'] in movie_idx.index:
            ref_indices.append(movie_idx[row['movieId']])
            ref_weights.append(row['rating'])
    ref_weights = np.array(ref_weights) if ref_weights else np.array([])
    scores = {}
    for mid in unseen:
        svd_p = model_svd.predict(uid, mid).est
        cb_p = 3.0
        if len(ref_indices) > 0 and mid in movie_idx.index:
            sims = cosine_sim[movie_idx[mid], ref_indices]
            cb_p = np.dot(sims, ref_weights) / len(ref_indices)
        scores[mid] = cf_weight * svd_p + (1 - cf_weight) * cb_p
    return [mid for mid, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]]

def eval_model(recs_fn, model_name, model=None):
    p5, p10, r5, r10, ap10_list, ndcg10_list = [], [], [], [], [], []
    for uid, actual in test_ground_truth.items():
        recs = recs_fn(model, uid) if model else recs_fn(uid)
        p5.append(precision_at_k(actual, recs, 5))
        p10.append(precision_at_k(actual, recs, 10))
        r5.append(recall_at_k(actual, recs, 5))
        r10.append(recall_at_k(actual, recs, 10))
        ap10_list.append(ap_at_k(actual, recs, 10))
        ndcg10_list.append(ndcg_at_k(actual, recs, 10))
    return {
        'precision@5': round(float(np.mean(p5)), 4), 'precision@10': round(float(np.mean(p10)), 4),
        'recall@5': round(float(np.mean(r5)), 4), 'recall@10': round(float(np.mean(r10)), 4),
        'map@10': round(float(np.mean(ap10_list)), 4), 'ndcg@10': round(float(np.mean(ndcg10_list)), 4),
    }

In [ ]:
print('Evaluating ranking metrics...')
ucf_r = eval_model(get_recs_surprise, 'User-CF', model_ucf); print('  ✅ User-CF')
icf_r = eval_model(get_recs_surprise, 'Item-CF', model_icf); print('  ✅ Item-CF')
svd_r = eval_model(get_recs_surprise, 'SVD', model_svd); print('  ✅ SVD')
cb_r = eval_model(get_recs_content, 'Content-Based'); print('  ✅ Content-Based')
hybrid_r = eval_model(get_recs_hybrid, 'Hybrid'); print('  ✅ Hybrid')
print('All evaluations done!')

## 7. Context Analysis

In [ ]:
merged = ratings.merge(users[['userId','gender','age_group','occupation_name']], on='userId')
merged = merged.merge(movies[['movieId','genres']], on='movieId')

gender_stats = {k: {'mean': round(float(v['mean']),4), 'count': int(v['count'])} for k,v in merged.groupby('gender')['rating'].agg(['mean','count']).to_dict('index').items()}

age_order = ['Under 18','18-24','25-34','35-44','45-49','50-55','56+']
age_stats = merged.groupby('age_group')['rating'].agg(['mean','count']).reindex(age_order).dropna()
age_stats_dict = {k: {'mean': round(float(v['mean']),4), 'count': int(v['count'])} for k,v in age_stats.iterrows()}

year_stats_dict = {int(k): {'mean': round(float(v['mean']),4), 'count': int(v['count'])} for k,v in merged.groupby('year')['rating'].agg(['mean','count']).iterrows()}

dow_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_stats_dict = {dow_names[int(k)]: round(float(v),4) for k,v in merged.groupby('dayofweek')['rating'].mean().items()}

all_genres = ['Action','Adventure','Animation',"Children's",'Comedy','Crime','Documentary','Drama','Fantasy','Film-Noir','Horror','Musical','Mystery','Romance','Sci-Fi','Thriller','War','Western']
for g in all_genres:
    merged[f'is_{g}'] = merged['genres'].str.contains(g, na=False).astype(int)
genre_gender = []
for g in all_genres:
    m = merged[(merged['gender']=='M') & (merged[f'is_{g}']==1)]['rating'].mean()
    f = merged[(merged['gender']=='F') & (merged[f'is_{g}']==1)]['rating'].mean()
    if pd.notna(m) and pd.notna(f):
        genre_gender.append({'genre': g, 'male': round(float(m),4), 'female': round(float(f),4), 'diff': round(float(f-m),4)})

with open('results/reports/context_analysis.json', 'w') as fp:
    json.dump({'rating_by_gender': gender_stats, 'rating_by_age': age_stats_dict, 'rating_by_year': year_stats_dict, 'rating_by_dayofweek': dow_stats_dict, 'genre_preference_by_gender': genre_gender}, fp, indent=2)

print('✅ Context analysis saved!')

## 8. Export All Results

In [ ]:
all_results = {
    'User-Based CF': {'RMSE': round(float(rmse_ucf),4), 'MAE': round(float(mae_ucf),4), **ucf_r},
    'Item-Based CF': {'RMSE': round(float(rmse_icf),4), 'MAE': round(float(mae_icf),4), **icf_r},
    'SVD': {'RMSE': round(float(rmse_svd),4), 'MAE': round(float(mae_svd),4), **svd_r},
    'Content-Based': {'RMSE': None, 'MAE': None, **cb_r},
    'Hybrid': {'RMSE': None, 'MAE': None, **hybrid_r},
}

with open('results/reports/evaluation_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

algos = list(all_results.keys())
rmse_summary = {
    'algorithms': algos,
    'rmse': [all_results[a].get('RMSE') for a in algos],
    'mae': [all_results[a].get('MAE') for a in algos],
    'precision_at_10': [all_results[a].get('precision@10') for a in algos],
    'recall_at_10': [all_results[a].get('recall@10') for a in algos],
    'map_at_10': [all_results[a].get('map@10') for a in algos],
    'ndcg_at_10': [all_results[a].get('ndcg@10') for a in algos],
    'best_by_rmse': min([a for a in algos if all_results[a].get('RMSE')], key=lambda a: all_results[a]['RMSE']),
    'best_by_map': max(algos, key=lambda a: all_results[a].get('map@10', 0)),
}
with open('results/reports/rmse_summary.json', 'w') as f:
    json.dump(rmse_summary, f, indent=2)

pd.DataFrame(all_results).T.to_csv('results/reports/final_evaluation.csv')

print('\n=== All results saved! ===')
print(json.dumps(rmse_summary, indent=2))

## 9. Visualize

In [ ]:
colors = ['#3B82F6','#6366F1','#10B981','#F59E0B','#EF4444']
algos = list(all_results.keys())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rmse_valid = [(a, v) for a, v in zip(algos, [all_results[a].get('RMSE') for a in algos]) if v]
axes[0].barh([a for a, _ in rmse_valid], [v for _, v in rmse_valid], color=colors[:len(rmse_valid)])
axes[0].set_xlabel('RMSE')
axes[0].set_title('RMSE Comparison')
for i, (_, v) in enumerate(rmse_valid):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center')

map_vals = [all_results[a].get('map@10', 0) for a in algos]
axes[1].barh(algos, map_vals, color=colors)
axes[1].set_xlabel('MAP@10')
axes[1].set_title('MAP@10 Comparison')
for i, v in enumerate(map_vals):
    axes[1].text(v + 0.001, i, f'{v:.4f}', va='center')

plt.tight_layout()
plt.savefig('results/charts/08_rmse_map_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

---

## 10. 🚀 Deploy — Colab API Server

Khởi động **FastAPI + ngrok** để frontend kết nối.

In [ ]:
!pip install fastapi uvicorn pyngrok joblib -q

In [ ]:
import joblib
os.makedirs('models', exist_ok=True)
joblib.dump(model_svd, 'models/svd_model.pkl')
joblib.dump(model_ucf, 'models/user_cf_model.pkl')
joblib.dump(model_icf, 'models/item_cf_model.pkl')
joblib.dump({'tfidf_matrix': tfidf_matrix, 'cosine_sim': cosine_sim, 'movie_idx': movie_idx}, 'models/content_based.pkl')
print('✅ Models saved!')

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import threading, uvicorn

api = FastAPI(title='Movie Recommender API', version='1.0')
api.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

class RecommendRequest(BaseModel):
    user_id: int
    algorithm: str = 'Hybrid'
    top_n: int = 10
    cf_weight: float = 0.6

@api.get('/api/health')
def health():
    return {'status': 'ok', 'models_loaded': True}

@api.post('/api/recommend')
def api_recommend(req: RecommendRequest):
    uid = req.user_id
    if uid not in ratings['userId'].values:
        raise HTTPException(404, f'User {uid} not found')
    algo = req.algorithm
    top_n = min(req.top_n, 50)
    if algo in ('SVD', 'User-Based CF', 'Item-Based CF'):
        model = {'SVD': model_svd, 'User-Based CF': model_ucf, 'Item-Based CF': model_icf}[algo]
        recs_ids = get_recs_surprise(model, uid, top_n)
    elif algo == 'Content-Based':
        recs_ids = get_recs_content(uid, top_n)
    elif algo == 'Hybrid':
        recs_ids = get_recs_hybrid(uid, req.cf_weight, top_n)
    else:
        raise HTTPException(400, f'Unknown algorithm: {algo}')
    results = []
    for mid in recs_ids:
        row = movies[movies['movieId'] == mid]
        if len(row) > 0:
            results.append({'movieId': int(mid), 'title': row.iloc[0]['title'], 'genres': row.iloc[0]['genres']})
    return {'user_id': uid, 'algorithm': algo, 'recommendations': results}

@api.get('/api/stats')
def api_stats():
    with open('results/reports/eda_stats.json') as f: return json.load(f)

@api.get('/api/evaluation')
def api_evaluation():
    with open('results/reports/rmse_summary.json') as f: return json.load(f)

print('✅ FastAPI app defined!')

In [ ]:
from pyngrok import ngrok

NGROK_TOKEN = '3BwEZnHe7Dx8Qqi8zIwT9UAAdSZ_4JVgkviSuc4ydxcE5z66S'

ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000)
print(f'\n🌐 API Public URL: {public_url}')
print(f'\n📋 Dán URL trên vào frontend config!')
print(f'\nTest: {public_url}/api/health')
print(f'Docs: {public_url}/docs')

def run_server():
    uvicorn.run(api, host='0.0.0.0', port=8000, log_level='warning')

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print('\n✅ Server đang chạy! Giữ notebook mở để server hoạt động.')